# OPS0077 p21/Rb/pRb cell embedding and cell-cycle clock feasibility

This notebook evaluates the 2D PCA embedding computed from p21, Rb, and pRb OrganelleProfiler mean-intensity features in the single-cell NTC dataset. It checks measurement coverage, latent-axis associations, and well-level technical structure. It then audits the available annotations and features for ground truth needed to train and validate a cell-cycle phase clock.

The input contains only non-targeting-control cells. Marker intensity and PCA position are not cell-cycle phase labels; p21/Rb/pRb alone do not establish G1, S, or G2/M. The notebook will not manufacture phase labels or report an unvalidated score as a clock. A supervised phase clock requires direct per-cell phase labels or experimentally grounded DNA-content labels.

Run from the project root. Tables and plots are written to `reports/ops0077_cell_cycle_clock/`.

## 1. Load the source and marker embedding

In [ ]:
from pathlib import Path
import json
import anndata as ad
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kruskal
import matplotlib.pyplot as plt

ROOT = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / 'data/altair/ops0077_4i/OPS0077_NTC_OrganelleProfiler_pRb_Rb_p21.h5ad').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Could not locate the project root containing the OPS0077 single-cell H5AD.')
DATA = ROOT / 'data/altair/ops0077_4i'
SOURCE = DATA / 'OPS0077_NTC_OrganelleProfiler_pRb_Rb_p21.h5ad'
EMBEDDING = DATA / 'OPS0077_NTC_OrganelleProfiler_pRb_Rb_p21_marker_pca.h5ad'
OUT = ROOT / 'reports/ops0077_cell_cycle_clock'
OUT.mkdir(parents=True, exist_ok=True)
adata = ad.read_h5ad(EMBEDDING, backed='r')
source = ad.read_h5ad(SOURCE, backed='r')
if not adata.obs_names.equals(source.obs_names):
    raise ValueError('Source and embedded observations are not in matching order.')
coords = np.asarray(adata.obsm['X_p21_Rb_pRb_pca'], dtype=np.float64)
obs = adata.obs.copy()
var = source.var.copy()
print(f'{adata.n_obs:,} cells × {adata.n_vars:,} features; embedding shape {coords.shape}')
print(f'Embedding-complete cells: {np.isfinite(coords).all(axis=1).sum():,}')
print(f'Output directory: {OUT}')

## 2. Check the three marker measurements and embedding coverage

The embedding was built from log1p-transformed, feature-wise standardized mean-intensity measurements. Cells missing any one marker are retained in the H5AD but have NaN embedding coordinates.

In [ ]:
marker_specs = {
    'pRb': '4i_r3_prb',
    'Rb': '4i_r3_rb',
    'p21': '4i_r4_p21',
}
feature_indices = {}
feature_rows = []
for marker, organelle in marker_specs.items():
    select = (
        var['source'].eq('organelle_profiler')
        & var['organelle'].eq(organelle)
        & var['category'].eq('intensity')
        & var['metric'].eq('intensity_mean')
        & var['aggregation'].eq('mean')
    )
    matches = np.flatnonzero(select.to_numpy())
    if len(matches) != 1:
        raise ValueError(f'Expected one {marker} intensity feature, found {len(matches)}')
    feature_indices[marker] = int(matches[0])
    feature_rows.append({'marker': marker, 'feature': str(source.var_names[matches[0]]), 'organelle': organelle})
marker_features = pd.DataFrame(feature_rows)
x = np.asarray(source.X[:, [feature_indices[m] for m in marker_specs]], dtype=np.float64)
marker_values = pd.DataFrame(x, index=obs.index, columns=list(marker_specs))
complete = np.isfinite(x).all(axis=1)
marker_features['cells_with_finite_values'] = [int(np.isfinite(x[:, j]).sum()) for j in range(x.shape[1])]
marker_features['missing_fraction'] = [float((~np.isfinite(x[:, j])).mean()) for j in range(x.shape[1])]
marker_features['feature'] = marker_features['feature'].astype(str)
display(marker_features)
coverage = pd.DataFrame({
    'measure': ['total cells', 'complete for all three markers', 'missing at least one marker', 'embedding coordinates finite'],
    'cells': [len(obs), int(complete.sum()), int((~complete).sum()), int(np.isfinite(coords).all(axis=1).sum())],
})
coverage['fraction_of_all_cells'] = coverage['cells'] / len(obs)
display(coverage)

## 3. Evaluate latent axes against their input markers

This checks that the PCA coordinates vary with the measurements used to construct them. These are descriptive correlations, not independent validation and not evidence that either axis represents a cell-cycle phase.

In [ ]:
valid = complete & np.isfinite(coords).all(axis=1)
log_markers = np.log1p(marker_values.loc[valid])
latent = pd.DataFrame(coords[valid], index=obs.index[valid], columns=['PC1', 'PC2'])
latent_var = adata.uns['marker_embedding']['explained_variance_ratio']
latent_summary = pd.DataFrame({
    'component': ['PC1', 'PC2'],
    'variance_explained': np.asarray(latent_var, dtype=float)[:2],
})
latent_summary['cumulative_variance_explained'] = latent_summary['variance_explained'].cumsum()
correlation_rows = []
for component in latent.columns:
    for marker in marker_values.columns:
        rho, pvalue = spearmanr(latent[component], log_markers[marker], nan_policy='omit')
        correlation_rows.append({'component': component, 'marker': marker, 'spearman_rho': rho, 'pvalue': pvalue})
marker_correlations = pd.DataFrame(correlation_rows)
display(latent_summary)
display(marker_correlations.pivot(index='marker', columns='component', values='spearman_rho'))
latent_summary.to_csv(OUT / 'embedding_variance.csv', index=False)
marker_correlations.to_csv(OUT / 'embedding_marker_correlations.csv', index=False)

## 4. Visualize the embedding and marker gradients

Marker gradients show how the measured inputs lie across the embedding. They must not be interpreted as G1/S/G2M assignments.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, marker in zip(axes, marker_values.columns):
    sc = ax.scatter(latent['PC1'], latent['PC2'], c=log_markers[marker], s=3, alpha=0.55, rasterized=True, cmap='viridis')
    ax.set_title(f'log1p {marker} intensity')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    fig.colorbar(sc, ax=ax, label='log1p intensity')
fig.suptitle('PCA embedding colored by input marker intensity')
fig.savefig(OUT / 'embedding_by_marker.png', dpi=180)
plt.show()

## 5. Check for well-level technical structure

The three well groups are tested descriptively with Kruskal–Wallis tests. A well association can indicate technical or sampling structure; it is not a biological cell-cycle effect.

In [ ]:
well_col = 'well_canonical' if 'well_canonical' in obs else 'well'
well_values = obs.loc[valid, well_col].astype(str)
technical_rows = []
for component in latent.columns:
    groups = [latent.loc[well_values.eq(level), component].to_numpy() for level in sorted(well_values.unique())]
    groups = [group for group in groups if len(group)]
    if len(groups) > 1:
        statistic, pvalue = kruskal(*groups)
        pooled = latent[component].to_numpy()
        grand = pooled.mean()
        between = sum(len(group) * (group.mean() - grand) ** 2 for group in groups)
        total = ((pooled - grand) ** 2).sum()
        technical_rows.append({'component': component, 'grouping': well_col, 'groups': len(groups), 'kruskal_statistic': statistic, 'pvalue': pvalue, 'eta_squared': between / total if total else np.nan})
technical = pd.DataFrame(technical_rows)
display(obs.loc[valid, well_col].astype(str).value_counts().rename_axis(well_col).to_frame('cells'))
display(technical)
technical.to_csv(OUT / 'well_association_tests.csv', index=False)
fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
for well in sorted(well_values.unique()):
    mask = well_values.eq(well).to_numpy()
    ax.scatter(latent.loc[mask, 'PC1'], latent.loc[mask, 'PC2'], s=3, alpha=0.35, label=well)
ax.set(xlabel='PC1', ylabel='PC2', title='Embedding by well')
ax.legend(title=well_col, markerscale=3)
fig.savefig(OUT / 'embedding_by_well.png', dpi=180)
plt.show()

## 6. Build embeddings from the per-cell channel feature blocks and channel relationships

This section uses the stored per-cell OrganelleProfiler features for pRb, Rb, and p21 (intensity, morphology, localization, and network measurements). Cross-channel CellProfiler colocalization is analyzed strictly as three independent two-channel blocks: pRb–Rb, pRb–p21, and Rb–p21. For each pair, PCA compares the two-channel feature block, that pair's colocalization features, and an integrated PCA of those two channels plus their pair-specific colocalization features. No three-channel colocalization block is pooled. DAPI-to-marker features are excluded.

Features with <80% finite coverage among cells with measurements for that block or no variation are excluded and documented. Remaining missing values are median-imputed, then features are median-centered, IQR-scaled, and clipped. PCA is fit to a seeded sample and projected across cells; blocks are scaled to balance their contribution.

In [ ]:
from scipy.stats import pearsonr

channel_organelle = {'pRb': '4i_r3_prb', 'Rb': '4i_r3_rb', 'p21': '4i_r4_p21'}
var_names = pd.Index(source.var_names.astype(str))
channel_indices = {
    marker: np.flatnonzero((var['source'].eq('organelle_profiler') & var['organelle'].eq(organelle)).to_numpy())
    for marker, organelle in channel_organelle.items()
}
pair_prefixes = {'pRb_Rb': 'cp_coloc_pRb_Rb_', 'pRb_p21': 'cp_coloc_pRb_p21_', 'Rb_p21': 'cp_coloc_Rb_p21_'}
pair_markers = {'pRb_Rb': ('pRb', 'Rb'), 'pRb_p21': ('pRb', 'p21'), 'Rb_p21': ('Rb', 'p21')}
pair_indices = {pair: np.flatnonzero(var_names.str.startswith(prefix)) for pair, prefix in pair_prefixes.items()}
print({marker: len(indices) for marker, indices in channel_indices.items()}, 'per-channel features')
print({pair: len(indices) for pair, indices in pair_indices.items()}, 'pair-specific colocalization features')
assert all(len(indices) == 221 for indices in channel_indices.values())
assert all(len(indices) == 16 for indices in pair_indices.values())

def prepare_feature_block(indices, group_name):
    raw = np.asarray(source.X[:, indices], dtype=np.float64)
    finite = np.isfinite(raw)
    available = finite.any(axis=1)
    coverage = finite[available].mean(axis=0)
    unique_counts = np.array([np.unique(raw[finite[:, j], j]).size for j in range(raw.shape[1])])
    retained = (coverage >= 0.80) & (unique_counts > 1)
    selected = raw[:, retained]
    selected_indices = np.asarray(indices)[retained]
    if selected.shape[1] == 0:
        raise ValueError(f'No features retained for {group_name}')
    selected_finite = np.isfinite(selected)
    medians = np.nanmedian(selected, axis=0)
    filled = np.where(selected_finite, selected, medians)
    q25, q75 = np.percentile(filled, [25, 75], axis=0)
    iqr = q75 - q25
    iqr[iqr == 0] = 1.0
    scaled = np.clip((filled - np.median(filled, axis=0)) / iqr, -10, 10).astype(np.float32)
    row_coverage = selected_finite.mean(axis=1)
    qc = pd.DataFrame({
        'group': group_name,
        'feature': var_names[np.asarray(indices)].to_numpy(),
        'finite_fraction': coverage,
        'unique_values': unique_counts,
        'retained': retained,
    })
    return scaled, row_coverage, selected_indices, qc

channel_blocks = {}
feature_qc_rows = []
for marker, indices in channel_indices.items():
    block = prepare_feature_block(indices, marker)
    channel_blocks[marker] = block
    feature_qc_rows.append(block[3])
pair_blocks = {}
for pair, indices in pair_indices.items():
    pair_blocks[pair] = prepare_feature_block(indices, f'{pair}_colocalization')
    feature_qc_rows.append(pair_blocks[pair][3])
feature_qc_all = pd.concat(feature_qc_rows, ignore_index=True)
feature_qc_all.to_csv(OUT / 'per_cell_feature_qc.csv', index=False)
display(feature_qc_all.groupby('group').agg(input_features=('feature', 'size'), retained_features=('retained', 'sum'), median_finite_fraction=('finite_fraction', 'median')))

channel_value_indices = {}
for marker, organelle in channel_organelle.items():
    candidate = var[(var['source'].eq('organelle_profiler')) & (var['organelle'].eq(organelle)) & (var['category'].eq('intensity')) & (var['metric'].eq('intensity_mean')) & (var['aggregation'].eq('mean'))]
    channel_value_indices[marker] = int(np.flatnonzero(var.index.isin(candidate.index))[0])
channel_values = pd.DataFrame(np.asarray(source.X[:, [channel_value_indices[m] for m in channel_organelle]], dtype=np.float64), index=obs.index, columns=list(channel_organelle))
pairwise_rows = []
for pair, markers in pair_markers.items():
    left, right = markers
    overlap = np.isfinite(channel_values[left]) & np.isfinite(channel_values[right])
    rho, p_spearman = spearmanr(channel_values.loc[overlap, left], channel_values.loc[overlap, right])
    r, p_pearson = pearsonr(channel_values.loc[overlap, left], channel_values.loc[overlap, right])
    pairwise_rows.append({'pair': pair, 'channel_1': left, 'channel_2': right, 'n_cells': int(overlap.sum()), 'spearman_rho': rho, 'spearman_pvalue': p_spearman, 'pearson_r': r, 'pearson_pvalue': p_pearson})
channel_correlations = pd.DataFrame(pairwise_rows)
display(channel_correlations)
channel_correlations.to_csv(OUT / 'channel_intensity_correlations.csv', index=False)
def fit_pca_and_project(matrix, valid_mask, seed=17, fit_cells=12000):
    rng = np.random.default_rng(seed)
    valid_indices = np.flatnonzero(valid_mask)
    if len(valid_indices) < 3:
        raise ValueError('Too few cells meet feature coverage for PCA.')
    fit_indices = rng.choice(valid_indices, size=min(fit_cells, len(valid_indices)), replace=False)
    fit_matrix = matrix[fit_indices].astype(np.float64)
    center = fit_matrix.mean(axis=0)
    fit_centered = fit_matrix - center
    covariance = (fit_centered.T @ fit_centered) / (len(fit_indices) - 1)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1][:2]
    eigenvalues = np.maximum(eigenvalues[order], 0)
    components = eigenvectors[:, order]
    coordinates = (matrix.astype(np.float64) - center) @ components
    coordinates[~valid_mask] = np.nan
    return coordinates.astype(np.float32), eigenvalues / max(float(np.trace(covariance)), 1e-12), len(fit_indices)
embedding_outputs = pd.DataFrame(index=obs.index)
pca_variance_rows = []
marker_coords = {}
channel_matrices = {}
channel_valid_masks = {}
for marker, (matrix, row_coverage, selected_indices, _) in channel_blocks.items():
    balanced_marker = matrix / np.sqrt(matrix.shape[1])
    marker_valid = row_coverage >= 0.80
    coordinates, marker_variance, n_fit = fit_pca_and_project(balanced_marker, marker_valid)
    embedding_outputs[[f'{marker}_PC1', f'{marker}_PC2']] = coordinates
    marker_coords[marker] = (coordinates, marker_valid)
    channel_matrices[marker] = balanced_marker
    channel_valid_masks[marker] = marker_valid
    pca_variance_rows.extend([{'pair': marker, 'model': f'{marker}_features', 'component': f'PC{i + 1}', 'variance_explained': float(value), 'fit_cells': n_fit, 'features': matrix.shape[1], 'valid_cells': int(marker_valid.sum())} for i, value in enumerate(marker_variance)])
pair_coordinates = {}
for pair, markers in pair_markers.items():
    left, right = markers
    pair_matrix, pair_row_coverage, pair_selected_indices, _ = pair_blocks[pair]
    pair_valid = pair_row_coverage >= 0.80
    pair_matrix_balanced = pair_matrix / np.sqrt(pair_matrix.shape[1])
    pair_channel_matrix = np.concatenate([channel_matrices[left], channel_matrices[right]], axis=1)
    pair_channel_valid = channel_valid_masks[left] & channel_valid_masks[right]
    channel_coords, channel_variance, n_fit = fit_pca_and_project(pair_channel_matrix, pair_channel_valid)
    pair_corr_coords, pair_corr_variance, n_fit_corr = fit_pca_and_project(pair_matrix_balanced, pair_valid)
    pair_integrated_valid = pair_channel_valid & pair_valid
    pair_integrated_matrix = np.concatenate([channel_matrices[left], channel_matrices[right], pair_matrix_balanced], axis=1)
    integrated_coords, integrated_variance, n_fit_integrated = fit_pca_and_project(pair_integrated_matrix, pair_integrated_valid)
    key = pair.lower()
    embedding_outputs[[f'{key}_channels_PC1', f'{key}_channels_PC2']] = channel_coords
    embedding_outputs[[f'{key}_coloc_PC1', f'{key}_coloc_PC2']] = pair_corr_coords
    embedding_outputs[[f'{key}_integrated_PC1', f'{key}_integrated_PC2']] = integrated_coords
    pair_coordinates[pair] = {'channels': (channel_coords, pair_channel_valid), 'colocalization': (pair_corr_coords, pair_valid), 'integrated': (integrated_coords, pair_integrated_valid)}
    pca_variance_rows.extend([{'pair': pair, 'model': model, 'component': f'PC{i + 1}', 'variance_explained': float(value), 'fit_cells': fit_count, 'features': feature_count, 'valid_cells': int(valid.sum())} for model, values, fit_count, feature_count, valid in [('two_channel_features', channel_variance, n_fit, pair_channel_matrix.shape[1], pair_channel_valid), ('pair_colocalization', pair_corr_variance, n_fit_corr, pair_matrix.shape[1], pair_valid), ('pair_integrated', integrated_variance, n_fit_integrated, pair_integrated_matrix.shape[1], pair_integrated_valid)] for i, value in enumerate(values)])
pca_variance = pd.DataFrame(pca_variance_rows)
pca_variance.to_csv(OUT / 'per_cell_pca_variance.csv', index=False)
embedding_outputs.insert(0, 'cell_id', obs.index.astype(str))
embedding_outputs.to_csv(OUT / 'per_cell_pca_embeddings.csv.gz', index=False)
display(pca_variance)
print({pair: int(result['integrated'][1].sum()) for pair, result in pair_coordinates.items()}, 'cells per pair-specific integrated PCA')

## 7. Compare the alternative projections

For each specified channel pair, compare PCA on the two channels' stored per-cell feature blocks, PCA on only that pair's colocalization features, and an integrated PCA of those same two channels plus their pair-specific colocalization. The three pair analyses remain separate; no three-channel correlation matrix is pooled. Plot points only when that model's input coverage passes the 80% threshold.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 14), constrained_layout=True)
for row, (pair, models) in enumerate(pair_coordinates.items()):
    for column, (model, (coordinates, mask)) in enumerate(models.items()):
        ax = axes[row, column]
        ax.scatter(coordinates[mask, 0], coordinates[mask, 1], s=3, alpha=0.35, rasterized=True)
        ax.set(title=f'{pair}: {model}', xlabel='PC1', ylabel='PC2')
fig.savefig(OUT / 'per_cell_pca_pairwise_comparison.png', dpi=180)
plt.show()

## 8. Audit cell-cycle labels and DNA-content measurements

DNA content is a planned independent evaluation endpoint, not a current feature or label. Search observation metadata for phase-like fields and feature names for direct DNA-content candidates. Generic nuclear intensity or DAPI-colocalization features are not treated as DNA content without assay documentation.

In [ ]:
phase_pattern = r'cell.?cycle|cell.?stage|dna.?content|ploidy'
phase_obs_cols = [str(col) for col in obs.columns if pd.Series([str(col)]).str.contains(phase_pattern, case=False, regex=True).iloc[0]]
phase_var_names = [str(name) for name in source.var_names if pd.Series([str(name)]).str.contains(phase_pattern, case=False, regex=True).iloc[0]]
dapi_var_names = [str(name) for name in source.var_names if 'dapi' in str(name).lower()]
predicted_class_counts = obs['predicted_class'].astype(str).value_counts(dropna=False).head(20).to_dict() if 'predicted_class' in obs else {}
direct_label_candidates = [name for name in phase_obs_cols if obs[name].notna().any()]
direct_dna_candidates = [name for name in dapi_var_names if any(token in name.lower() for token in ['dna_content', 'integrated_dna', 'dapi_int', 'dapi_mean'])]
clock_audit = {
    'observation_count': int(len(obs)),
    'all_cells_non_targeting_control': bool(obs['control_type'].astype(str).eq('non-targeting').all()) if 'control_type' in obs else None,
    'phase_like_observation_columns': phase_obs_cols,
    'phase_or_dna_feature_names': phase_var_names,
    'dapi_feature_names': dapi_var_names,
    'direct_phase_label_candidates': direct_label_candidates,
    'direct_dna_content_candidates': direct_dna_candidates,
    'predicted_class_value_counts': predicted_class_counts,
    'supervised_phase_clock_trainable': bool(direct_label_candidates),
}
print(json.dumps(clock_audit, indent=2, default=str))
pd.DataFrame([{'candidate_type': 'phase_obs_column', 'candidate': name} for name in phase_obs_cols] + [{'candidate_type': 'phase_or_DNA_feature', 'candidate': name} for name in phase_var_names] + [{'candidate_type': 'DAPI_feature', 'candidate': name} for name in dapi_var_names]).to_csv(OUT / 'phase_label_feature_audit.csv', index=False)

## 7. Cell-cycle clock decision and next validation requirements

A supervised G1/S/G2M predictor cannot be trained or evaluated from this dataset: it contains only NTCs, the predicted-class field is empty, and no phase ground truth or documented DNA-content measurement is available. Consequently, this notebook does not assign phases or train a classifier. The 2D PCA is an exploratory marker-state embedding, not a cell-cycle clock.

DNA content is planned as an independent evaluation endpoint once available; it is not needed to compute the current marker embeddings. Until then, the embeddings and channel-correlation features remain exploratory. A phase clock still requires trusted phase labels or externally defined phase bins, and should be validated by well/plate and biological replicate with per-phase metrics and calibrated probabilities. NTC-only data cannot establish perturbation effects or a supervised phase decision boundary.

In [ ]:
clock_audit['embedding_method'] = str(adata.uns['marker_embedding']['method'])
clock_audit['embedding_dimensions'] = int(adata.uns['marker_embedding']['dimensions'])
clock_audit['embedding_complete_cells'] = int(valid.sum())
clock_audit['embedding_variance_explained'] = [float(value) for value in np.asarray(latent_var, dtype=float)[:2]]
clock_audit['decision'] = 'No phase clock trained: no per-cell phase labels or documented direct DNA-content measurements; NTC-only sample.'
clock_audit['latent_axis_marker_correlations'] = marker_correlations.to_dict(orient='records')
clock_audit['well_association_tests'] = technical.to_dict(orient='records')
clock_audit['per_cell_feature_counts'] = feature_qc_all.groupby('group')['retained'].sum().astype(int).to_dict()
clock_audit['channel_intensity_correlations'] = channel_correlations.to_dict(orient='records')
clock_audit['per_cell_pca_variance'] = pca_variance.to_dict(orient='records')
clock_audit['pairwise_integrated_pca_cells'] = {pair: int(result['integrated'][1].sum()) for pair, result in pair_coordinates.items()}
clock_audit['dna_content_status'] = 'Not present; reserved as an independent future validation endpoint.'
(OUT / 'analysis_summary.json').write_text(json.dumps(clock_audit, indent=2, default=str))
print(f"Saved evaluation tables, plots, and clock-feasibility decision to {OUT}")